In [24]:
import nest_asyncio
nest_asyncio.apply()  # Required for Gremlin driver in Jupyter (avoids "event loop already running" error)

from pathlib import Path
from dotenv import load_dotenv

# Load .env from agents folder
env_path = Path.cwd() / '.env'
if not env_path.exists():
    env_path = Path.cwd() / 'backend' / 'agents' / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✓ Loaded .env from: {env_path}")
else:
    print("⚠ .env not found in agents folder")

✓ Loaded .env from: /Users/rob.bajra/Projects/voice-to-rag/backend/agents/.env


In [25]:
# PuppyGraph Client setup
import os
from simple_client import PuppyGraphClient

client = PuppyGraphClient(username=os.getenv("PUPPYGRAPH_USERNAME"), password=os.getenv("PUPPYGRAPH_PASSWORD"))
client.connect()

client.query("g.V().limit(10)")



[v[{'@type': 'g:String', '@value': 'table[00vsdb.agent_analytics.agents_clean]'}],
 v[{'@type': 'g:String', '@value': 'table[00vsdb.default.student_silver]'}],
 v[{'@type': 'g:String', '@value': 'table[00vsdb.mlops.openai_payload]'}],
 v[{'@type': 'g:String', '@value': 'table[03_31_2025_test_cs.dbdemos_aibi_customer_support.agents_bronze]'}],
 v[{'@type': 'g:String', '@value': 'table[03_34_2025_test_cs.dbdemos_aibi_cme_marketing_campaign.feedbacks]'}],
 v[{'@type': 'g:String', '@value': 'table[04_01_cal_test_aibi_customer_support.dbdemos_aibi_customer_support.agents_bronze]'}],
 v[{'@type': 'g:String', '@value': 'table[04_29_2025_test_cal_claims.dbdemos_fsi_smart_claims.__materialization_mat_624bbb00_aa4d_499b_b4d9_e0313e828d3d_claim_policy_telematics_1]'}],
 v[{'@type': 'g:String', '@value': 'table[04_29_2025_test_cal_claims.dbdemos_fsi_smart_claims.claim_policy_telematics]'}],
 v[{'@type': 'g:String', '@value': 'table[04_29_2025_test_cal_claims.dbdemos_fsi_smart_claims.damage_predict

In [26]:

# Databricks model setup (E2 demo workspace)
from openai import OpenAI

ai_client = OpenAI(
  api_key=os.getenv("DATABRICKS_TOKEN"), # your personal access token
  base_url=os.getenv("DATABRICKS_HOST") + "serving-endpoints", # your Databricks workspace instance
)


In [27]:
# PuppyGraph tools (reload to pick up changes)
import importlib
import cypher_tool
importlib.reload(cypher_tool)

from cypher_tool import (
    GENERATE_CYPHER_TOOL_SPEC, generate_cypher,
    EXECUTE_CYPHER_TOOL_SPEC, make_execute_cypher,
)

# Bind execute_cypher to the PuppyGraph client from Cell 3
execute_cypher = make_execute_cypher(client)

# Tool specs (dicts for the OpenAI API) and local functions
TOOLS = [GENERATE_CYPHER_TOOL_SPEC, EXECUTE_CYPHER_TOOL_SPEC]



In [ ]:
import json

# Map tool names to their local Python functions
TOOL_FUNCTIONS = {
    "generate_cypher": generate_cypher,
    "execute_cypher": execute_cypher,
}

LLM_ENDPOINT_NAME = "databricks-gemini-3-pro"

messages = [
    {
        "role": "system",
        "content": """
        You are a Puppygraph agent. 
        Use the generate_cypher tool to convert natural language questions into cypher queries.
        Use the execute_cypher tool to execute cypher queries on the graph database.
        
        """,
    },
    {
        "role": "user",
        "content": "List all tables in the database. limit to 10",
    },
]

# Step 1: Send request with tool specs
response = ai_client.chat.completions.create(
    messages=messages,
    model=LLM_ENDPOINT_NAME,
    tools=TOOLS,
    max_tokens=256,
)

choice = response.choices[0]

# Step 2: If the model wants to call tool(s), execute them and send results back
if choice.finish_reason == "tool_calls":
    # Add the assistant's tool-call message to history
    messages.append(choice.message)

    for tool_call in choice.message.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        print(f"Tool call: {fn_name}({fn_args})")

        # Execute the tool locally
        result = TOOL_FUNCTIONS[fn_name](**fn_args)
        print(f"Tool result: {result}\n")

        # Add tool result to message history
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        })

    # Step 3: Send tool results back to get final answer
    response = ai_client.chat.completions.create(
        messages=messages,
        model=LLM_ENDPOINT_NAME,
        tools=TOOLS,
        max_tokens=256,
    )
    print("Final answer:", response.choices[0].message.content)
else:
    # Model answered directly without calling a tool
    print(response.choices[0].message.content)


Tool call: generate_cypher({'cypher_query': 'MATCH (t:table) RETURN id(t) LIMIT 10'})
Tool result: MATCH (t:table) RETURN id(t) LIMIT 10

Final answer: MATCH (t:table) RETURN id(t) LIMIT 10


In [17]:
print(chat_completion.choices[0].message.content)


None


In [ ]:
# Test generate_cypher tool directly (natural language -> Cypher)

cypher = generate_cypher("What is the lineage of table 00vsdb.agent_analytics.agents_clean?")
print(cypher)

What is the lineage of table 00vsdb.agent_analytics.agents_clean?


## Find lineage of a given table